In [ ]:
# Enables IPython autoreload (two magic commands, the second takes a numeric argument)
%load_ext autoreload
%autoreload 2

import logging
import os
import time

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s  - %(name)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", "{:.6f}".format)


logger.info("Notebook initialized")

---

## Final pipeline using `src/data.py`

The above cells walked through the exploration step-by-step. The same pipeline is now packaged into two functions in `src/data.py`. The cells below verify that the imported functions produce the same output as the inline exploration.

In [ ]:
from src.data import compute_returns, download_prices

prices = download_prices()
returns = compute_returns(prices)

print("Daily prices shape:", prices.shape)
print("Monthly returns shape:", returns.shape)
print("\nFirst 3 returns:")
print(returns.head(3))
print("\nLast 3 returns:")
print(returns.tail(3))

In [ ]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, geometric_mean

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nGeometric mean (monthly):")
print(geometric_mean(returns))

In [ ]:
from src.data import compute_returns, download_prices
from src.stats import arithmetic_mean, covariance_matrix, geometric_mean, volatility

prices = download_prices()
returns = compute_returns(prices)

print("Arithmetic mean (monthly):")
print(arithmetic_mean(returns))
print("\nVolatility (monthly):")
print(volatility(returns))
print("\nCovariance matrix (monthly):")
print(covariance_matrix(returns))
print("\nDiagonal of covariance vs volatility squared:")
print((volatility(returns) ** 2).round(8))
print(np.diag(covariance_matrix(returns)).round(8))

In [ ]:
from src.data import compute_returns, download_prices
from src.optimizer import solve_mvo_scipy
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

# Try a target return that should be feasible: middle of the per-asset range
target = 0.01  # 1% monthly target

weights = solve_mvo_scipy(mu, sigma, target_return=target)
print("Weights:")
print(weights)
print(f"\nSum of weights: {weights.sum():.6f}")
print(f"Portfolio expected return: {(mu * weights).sum():.6f}")
print(f"Portfolio variance: {weights @ sigma @ weights:.6f}")
print(f"Portfolio volatility: {(weights @ sigma @ weights)**0.5:.6f}")

In [ ]:
from src.data import compute_returns, download_prices
from src.optimizer import solve_mvo_cvxpy, solve_mvo_scipy
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

target = 0.01
weights_scipy = solve_mvo_scipy(mu, sigma, target_return=target)
weights_cvxpy = solve_mvo_cvxpy(mu, sigma, target_return=target)

print("scipy weights:")
print(weights_scipy.round(4))
print("\ncvxpy weights:")
print(weights_cvxpy.round(4))
print("\nMax absolute difference:")
print((weights_scipy - weights_cvxpy).abs().max())

In [ ]:
from src.data import compute_returns, download_prices
from src.optimizer import solve_mvo_cvxpy, solve_mvo_scipy
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

target = 0.01
ws = solve_mvo_scipy(mu, sigma, target_return=target)
wc = solve_mvo_cvxpy(mu, sigma, target_return=target)

# Compute variance of each portfolio under the same Σ
var_s = ws @ sigma @ ws
var_c = wc @ sigma @ wc

print(f"scipy variance:  {var_s:.10f}")
print(f"cvxpy variance:  {var_c:.10f}")
print(f"scipy return:    {(mu * ws).sum():.10f}")
print(f"cvxpy return:    {(mu * wc).sum():.10f}")
print(f"scipy sum:       {ws.sum():.10f}")
print(f"cvxpy sum:       {wc.sum():.10f}")

In [ ]:
import numpy as np

from src.data import compute_returns, download_prices
from src.frontier import efficient_frontier
from src.stats import arithmetic_mean, covariance_matrix

prices = download_prices()
returns = compute_returns(prices)
mu = arithmetic_mean(returns)
sigma = covariance_matrix(returns)

targets = np.linspace(0.001, 0.014, 14)  # avoid the infeasible high targets

f1 = efficient_frontier(mu, sigma, targets, min_weight=-1.0, max_weight=1.0)
f2 = efficient_frontier(mu, sigma, targets, min_weight=0.0, max_weight=1.0)
f3 = efficient_frontier(mu, sigma, targets, min_weight=0.05, max_weight=1.0)

print(
    f"Regime 1 (short OK):    {len(f1)} points, vol range [{f1['volatility'].min():.4f}, {f1['volatility'].max():.4f}]"
)
print(
    f"Regime 2 (long-only):   {len(f2)} points, vol range [{f2['volatility'].min():.4f}, {f2['volatility'].max():.4f}]"
)
print(
    f"Regime 3 (min 5%):      {len(f3)} points, vol range [{f3['volatility'].min():.4f}, {f3['volatility'].max():.4f}]"
)

common = f1.index.intersection(f2.index).intersection(f3.index)
v1 = f1.loc[common, "volatility"]
v2 = f2.loc[common, "volatility"]
v3 = f3.loc[common, "volatility"]

monotonic = ((v1 <= v2 + 1e-10) & (v2 <= v3 + 1e-10)).all()
print(f"\nMonotonicity (vol1 <= vol2 <= vol3) on {len(common)} common targets: {monotonic}")